# Train an image classifier with TIMM (GeoAI)

This notebook replicates the **GeoAI** walkthrough [*Train timm classifier*](https://opengeoai.org/examples/train_timm_classifier/), which shows how to fine-tune **PyTorch Image Models** ([timm](https://github.com/huggingface/pytorch-image-models)) on image classification—including remote-sensing style pipelines—using the high-level API in `geoai.timm_train`.

**Original tutorial:** [opengeoai.org/examples/train_timm_classifier/](https://opengeoai.org/examples/train_timm_classifier/)


## Key ideas

- **Large model zoo:** timm exposes many pretrained backbones (ResNet, EfficientNet, ViT, ConvNeXt, …).
- **Multi-channel inputs:** You can set `in_channels` (and the dataset’s channel count) for RGB, RGB+NIR, etc. (see `train_timm_classifier`).
- **Lightning training loop:** Checkpointing, CSV logs, and early stopping are handled by PyTorch Lightning.
- **Transfer learning:** Load ImageNet weights (`pretrained=True`) and optionally **freeze the backbone** to train only the head first.

`RemoteSensingDataset` in GeoAI loads rasters with **rasterio** (paths must be readable by your GDAL/rasterio build, including many JPEG/PNG setups).

## Install packages

Install GeoAI, timm, Lightning, and Hugging Face `datasets` (for EuroSAT in this demo). Rasterio is pulled in as needed for reading image files.


In [ ]:
! uv pip install geoai-py timm lightning datasets

## Imports


In [ ]:
import os
import tempfile

from geoai.timm_train import (
    RemoteSensingDataset,
    TimmClassifier,
    list_timm_models,
    predict_with_timm,
)

# This class has a bug that I fixed manually.
# Original class from geoai has a bug using logger on line 523.
from helpers.timm_train import train_timm_classifier

## Explore available timm models

Filter names to browse architectures before picking `model_name` for training.


In [ ]:
resnet_models = list_timm_models(filter="resnet", limit=10)
print("ResNet models:", resnet_models)


In [ ]:
efficientnet_models = list_timm_models(filter="efficientnet", limit=10)
print("EfficientNet models:", efficientnet_models)


In [ ]:
vit_models = list_timm_models(filter="vit", limit=10)
print("Vision Transformer models:", vit_models)


## Download sample data (EuroSAT RGB)

The tutorial uses the **EuroSAT RGB** dataset from Hugging Face ([`timm/eurosat-rgb`](https://huggingface.co/datasets/timm/eurosat-rgb)): Sentinel-2–style RGB chips over **10** land-cover classes:

- AnnualCrop, Forest, HerbaceousVegetation, Highway, Industrial, Pasture, PermanentCrop, Residential, River, SeaLake

Images are written to a temporary folder in **class subdirectories** (ImageFolder layout) so we can build path lists and integer labels.


In [ ]:
from datasets import load_dataset
from PIL import Image

print("Loading EuroSAT from Hugging Face...")
dataset = load_dataset("timm/eurosat-rgb")

train_temp_dir = tempfile.mkdtemp(prefix="eurosat_train")
print(f"Saving images to: {train_temp_dir}")

test_temp_dir = tempfile.mkdtemp(prefix="eurosat_test")
print(f"Saving images to: {test_temp_dir}")

val_temp_dir = tempfile.mkdtemp(prefix="eurosat_val")
print(f"Saving images to: {val_temp_dir}")

class_names = dataset["train"].features["label"].names

print("Classes:", class_names)

for dataset, temp_dir in zip([dataset["train"], dataset["test"], dataset["validation"]], [train_temp_dir, test_temp_dir, val_temp_dir]):
    for idx, sample in enumerate(dataset):
        img = sample["image"]
        label = sample["label"]
        class_name = class_names[label]
        class_dir = os.path.join(temp_dir, class_name)
        os.makedirs(class_dir, exist_ok=True)
        img_path = os.path.join(class_dir, f"{idx:05d}.jpg")
        img.save(img_path)

print(f"Saved {len(dataset)} images to {temp_dir}")


## Prepare file lists and labels

Collect paths in class order, assign **integer labels** `0 … num_classes-1`, then print class counts.


In [ ]:
import glob

from sklearn.model_selection import train_test_split

image_paths = {
    "train": [],
    "test": [],
    "val": []
}

labels = {
    "train": [],
    "test": [],
    "val": []
}

for t,temp_dir in zip(["train", "test", "val"], [train_temp_dir, test_temp_dir, val_temp_dir]):
    for class_idx, class_name in enumerate(class_names):
        class_dir = os.path.join(temp_dir, class_name)
        class_images = sorted(glob.glob(os.path.join(class_dir, "*.jpg")))
        image_paths[t].extend(class_images)
        labels[t].extend([class_idx] * len(class_images))

    print(f"Dataset: {t}")
    print(f"Total images: {len(image_paths[t])}")
    print(f"Number of classes: {len(class_names)}")
    print("Class distribution:")
    for class_idx, class_name in enumerate(class_names):
        count = labels[t].count(class_idx)
        print(f"  {class_name}: {count}")


## Show class distribution

In [ ]:
print("Plot class distribution and balance")

import matplotlib.pyplot as plt
import numpy as np

# Count samples per class
for t in ["train", "test", "val"]:
    counts = [labels[t].count(i) for i in range(len(class_names))]

    plt.figure(figsize=(12, 6))
    bars = plt.bar(class_names, counts, color='skyblue')
    plt.title("Class Distribution in EuroSAT Dataset")
    plt.xlabel("Class")
    plt.ylabel("Number of images")
    plt.xticks(rotation=25, ha='right')

    # Annotate bars with counts
    for bar, count in zip(bars, counts):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5, str(count),
                ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.show()

    print("Mean images per class:", np.mean(counts))
    print("Standard deviation:", np.std(counts))

## Train / validation / test splits

Stratified split: **20%** held out as test; from the remaining **80%**, take **20%** of that as validation (tutorial convention). Adjust fractions as needed for your project.


In [ ]:
# train_paths, test_paths, train_labels, test_labels = train_test_split(
#     image_paths,
#     labels,
#     test_size=0.2,
#     random_state=42,
#     stratify=labels,
# )

# train_paths, val_paths, train_labels, val_labels = train_test_split(
#     train_paths,
#     train_labels,
#     test_size=0.2,
#     random_state=42,
#     stratify=train_labels,
# )

# print(f"Training samples: {len(train_paths)}")
# print(f"Validation samples: {len(val_paths)}")
# print(f"Test samples: {len(test_paths)}")


## Visualize one example per class

Quick sanity check of labels and appearance before training.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for idx, class_name in enumerate(class_names):
    ax = axes[idx // 5, idx % 5]
    img_idx = labels["train"].index(idx)
    img = Image.open(image_paths["train"][img_idx])
    ax.imshow(img)
    ax.set_title(class_name, fontsize=12)
    ax.axis("off")

plt.tight_layout()
plt.show()


## Build `RemoteSensingDataset` objects

`num_channels=3` matches RGB. The loader returns tensors shaped **(C, H, W)** in float approximately **\[0, 1\]** after normalization in the dataset implementation.


In [ ]:
train_dataset = RemoteSensingDataset(
    image_paths=image_paths["train"],
    labels=labels["train"],
    num_channels=3,
)

val_dataset = RemoteSensingDataset(
    image_paths=image_paths["val"],
    labels=labels["val"],
    num_channels=3,
)

test_dataset = RemoteSensingDataset(
    image_paths=image_paths["test"],
    labels=labels["test"],
    num_channels=3,
)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")


## Train a ResNet-50 classifier

Fine-tune **ResNet-50** with ImageNet weights on the 10 EuroSAT classes. Training runs for several epochs; reduce `num_epochs` for a quick smoke test.

**Checkpoints** land under `output_dir/models/` (including `last.ckpt`). Monitoring `val_acc` with `mode="max"` matches the tutorial’s accuracy-focused early stopping.


In [ ]:
# ! uv pip install --upgrade timm  geoai-py pip pytorch-lightning


In [ ]:
output_dir = "timm_output/resnet50"
if not os.path.exists(output_dir):
    model = train_timm_classifier(
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        test_dataset=test_dataset,
        model_name="resnet50",
        num_classes=len(class_names),
        in_channels=3,
        pretrained=True,
        output_dir=output_dir,
        batch_size=32,
        num_epochs=20,
        learning_rate=1e-3,
        weight_decay=1e-4,
        num_workers=4,
        freeze_backbone=False,
        monitor_metric="val_acc",
        mode="max",
        patience=5,
        save_top_k=1,
    )


## Train EfficientNet-B0

Smaller footprint than ResNet-50; often a good accuracy–compute tradeoff.


In [ ]:
output_dir = "timm_output/efficientnet_b3"
if not os.path.exists(output_dir):
    model = train_timm_classifier(
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        test_dataset=test_dataset,
        model_name="efficientnet_b3",
        num_classes=len(class_names),
        in_channels=3,
        pretrained=True,
        output_dir=output_dir,
        batch_size=32,
        num_epochs=20,
        learning_rate=1e-3,
        weight_decay=1e-4,
        num_workers=4,
        freeze_backbone=False,
        monitor_metric="val_acc",
        mode="max",
        patience=5,
        save_top_k=1,
    )


## Fine-tune with a frozen backbone

Set `freeze_backbone=True` so only the **classification head** updates (faster, less risk of wrecking low-level filters on tiny datasets). The GeoAI module freezes parameters whose names do not contain `fc`, `head`, or `classifier`.


In [ ]:
output_dir = "timm_output/resnet50_frozen"
if not os.path.exists(output_dir):
    model_frozen = train_timm_classifier(
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        test_dataset=test_dataset,
        model_name="resnet50",
        num_classes=len(class_names),
        in_channels=3,
        pretrained=True,
        freeze_backbone=True,
        output_dir=output_dir,
        batch_size=32,
        num_epochs=10,
        learning_rate=1e-3,
        monitor_metric="val_acc",
        mode="max",
    )


## Run inference with `predict_with_timm`

Load a saved Lightning checkpoint with `TimmClassifier.load_from_checkpoint`, then batch images from the test list. Set `return_probabilities=True` to get softmax scores for error analysis or calibration.


In [ ]:
from datetime import datetime

checkpoint_paths =[
    "timm_output/resnet50/models/last.ckpt",
    "timm_output/efficientnet_b3/models/last.ckpt",
    "timm_output/resnet50_frozen/models/last.ckpt",
]

models = {}
for checkpoint_path in checkpoint_paths:
    model_name = checkpoint_path.split("/")[1]
    model = TimmClassifier.load_from_checkpoint(checkpoint_path)

    start_time = datetime.now()

    predictions, probabilities = predict_with_timm(
        model=model,
        image_paths=image_paths["val"][:20],
        batch_size=8,
        return_probabilities=True,
    )

    end_time = datetime.now()
    inference_time = end_time - start_time
    models[model_name] = {
        "predictions": predictions,
        "probabilities": probabilities,
        "model": model,
        "inference_time": inference_time
    }

    print(f"Predictions shape: {predictions.shape}")
    print(f"Probabilities shape: {probabilities.shape}")
    print(f"Sample predictions: {[class_names[p] for p in predictions[:5]]}")

for model_name, model_info in models.items():
    print("="*50)
    print(f"Model parameters: {sum(p.numel() for p in model_info['model'].parameters())}")
    print(f"Model: {model_name}")
    print(f"Inference time: {model_info['inference_time']}")


In [ ]:
!du -shc timm_output/*

## Visualize predictions on a batch

Green titles: predicted class matches the test label; red: mismatch. Confidence is the probability assigned to the argmax class.


In [ ]:
def plot_predictions(predictions, probabilities, test_paths, test_labels, class_names):
    fig, axes = plt.subplots(4, 5, figsize=(20, 16))

    for idx, ax in enumerate(axes.flat):
        if idx >= len(test_paths[:20]):
            break
        img = Image.open(test_paths[idx])
        ax.imshow(img)
        pred_class = class_names[predictions[idx]]
        true_class = class_names[test_labels[idx]]
        confidence = probabilities[idx][predictions[idx]] * 100
        color = "green" if predictions[idx] == test_labels[idx] else "red"
        ax.set_title(
            f"Pred: {pred_class}\nTrue: {true_class}\n({confidence:.1f}%)",
            color=color,
            fontsize=10,
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()


# Pred Res50

In [ ]:
plot_predictions(models["resnet50"]["predictions"], models["resnet50"]["probabilities"], image_paths["val"], labels["val"], class_names)

In [ ]:
plot_predictions(models["efficientnet_b3"]["predictions"], models["efficientnet_b3"]["probabilities"], image_paths["val"], labels["val"], class_names)

In [ ]:
plot_predictions(models["resnet50_frozen"]["predictions"], models["resnet50_frozen"]["probabilities"], image_paths["val"], labels["val"], class_names)

## First conclusion
<p>
We trained three different models. We applied transfer learning to both the ResNet50 and ConvNeXt architectures, and additionally fine-tuned a ResNet50 model with a frozen backbone.

The validation dataset indicates that the models perform well.

*However, what happens when we run inference on external images outside the training distribution? How do factors such as image quality, object distance, and perspective affect model performance?*
</p>

In [ ]:
satellite_custom_images = "media/dataset/timm/hq"
temp_path = "/tmp/timm"

os.makedirs(temp_path, exist_ok=True)

images_path = os.listdir(satellite_custom_images)
print(f"Ploting original {len(images_path)} images...")

n_axes = 2
n_rows_calc = int(len(images_path)/n_axes)
n_rows = 1 if n_rows_calc == 0 else n_rows_calc
fig, axes = plt.subplots(n_rows, n_axes, figsize=(20, 16))
idx = 0

for image_path in images_path:
    if image_path.startswith("."):
        continue

    image = Image.open(f"{satellite_custom_images}/{image_path}")

    if n_rows > 1:
        ax = axes[idx // n_axes, idx % n_axes]
    else:
        ax = axes[idx]
    idx += 1
    ax.imshow(image)
    ax.set_title(
        f"{image_path}",
        fontsize=10,
    )
    ax.axis("off")


In [ ]:
print("Cropping images in batches of 64x64 pixels...")
def crop_image_batch(image, size=(64, 64)):

    width, height = image.size
    for i in range(0, width, size[0]):
        for j in range(0, height, size[1]):
            yield image.crop((i, j, i + size[0], j + size[1]))

cropped_images = {}
for image_path in images_path:
    image_name = image_path.split(".")[0]
    image = Image.open(f"{satellite_custom_images}/{image_path}")
    cropped_images[image_name] = list(crop_image_batch(image))
    print(f"Cropped {image_name} {len(cropped_images[image_name])} images...")




In [ ]:

import random

cropped_path = f"{temp_path}/cropped-hq"
os.makedirs(cropped_path, exist_ok=True)

for image_name, imgs in cropped_images.items():
    n_axes = 5
    n_rows = 3
    fig, axes = plt.subplots(n_rows, n_axes, figsize=(20, 16))

    plt.title(f"{image_name}", fontsize=10)
    random.shuffle(imgs)

    for idx, ax in enumerate(axes.flat):
        if idx >= len(imgs):
            break
        ax.imshow(imgs[idx])
        #ax.axis("off")
        imgs[idx] = imgs[idx].convert("RGB")
        imgs[idx].save(f"{cropped_path}/{image_name}_{idx}.png")

    plt.tight_layout()
print(f"Cropped path: {cropped_path}")

In [ ]:
def plot_predictions(predictions, probabilities, images_path, class_names):
    n_axes = 5
    n_rows = len(images_path) // n_axes
    fig, axes = plt.subplots(n_rows, n_axes, figsize=(20, 16))

    for idx, ax in enumerate(axes.flat):
        if idx >= len(images_path):
            break
        image = Image.open(images_path[idx])
        ax.imshow(image)
        pred_class = class_names[predictions[idx]]
        confidence = probabilities[idx][predictions[idx]] * 100
        color = "green" if confidence >50 else "red"
        ax.set_title(
            f"Pred: {pred_class}\n ({confidence:.1f}%)",
            color=color,
            fontsize=10,
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
print("Predicting Resnet50...")

cropped_images_paths = [f"{cropped_path}/{image_path}" for image_path in os.listdir(cropped_path)]

predictions, probabilities = predict_with_timm(
    model=models["resnet50"]["model"],
    image_paths=cropped_images_paths,
    batch_size=8,
    return_probabilities=True,
)

print(f"Predictions: {predictions}")


In [ ]:
plot_predictions(predictions, probabilities, cropped_images_paths, class_names)

In [ ]:
print("Predicting efficientnet_b3...")

predictions, probabilities = predict_with_timm(
    model=models["efficientnet_b3"]["model"],
    image_paths=cropped_images_paths,
    batch_size=8,
    return_probabilities=True,
)

print(f"Predictions: {predictions}")

plot_predictions(predictions, probabilities, cropped_images_paths, class_names)

## Evaluating Model Performance Without Ground Truth

Is it possible to assess a model’s performance without access to ground truth labels?

While quantitative evaluation becomes limited, qualitative analysis can still provide valuable insights. By visually inspecting the model’s predictions, we can identify patterns in its behavior. In this case, the model demonstrates strong performance when detecting residential areas, but its accuracy noticeably declines when identifying highways, rivers, and lakes.

---

## Observations from Prediction Analysis

* **High performance**: Residential areas (well-defined patterns, consistent textures)
* **Lower performance**: Highways, rivers, and lakes (more variability, less distinctive features)

This discrepancy often indicates:

* Class imbalance in the training dataset
* Insufficient feature representation for underperforming classes
* Sensitivity to variations such as scale, lighting, or viewpoint

---

## How to Improve Performance for Underperforming Classes

### 1. Data-Centric Improvements

* **Increase dataset diversity** for highways, rivers, and lakes
* **Balance class distribution** to reduce bias toward dominant classes
* Apply **targeted data augmentation**:

  * Vary brightness, contrast, and resolution
  * Simulate different perspectives and distances
  * Introduce noise and blur to improve robustness

### 2. Label Quality and Sampling

* Ensure annotations are **accurate and consistent**
* Use **hard example mining** to focus training on difficult samples
* Oversample underrepresented classes or apply **class-weighted loss functions**

### 3. Model and Training Strategy

* Fine-tune deeper layers instead of keeping the backbone fully frozen
* Experiment with **loss functions** such as Focal Loss to handle class imbalance
* Use **multi-scale training** to improve detection across different object sizes

---

## Generating Ground Truth for the Dataset

If ground truth is not available, it must be created or approximated:

### 1. Manual Annotation

* Use tools such as:

  * LabelImg
  * CVAT
  * Label Studio
* Most accurate approach, but time-consuming

### 2. Semi-Automatic Labeling

* Use the current model to generate **pseudo-labels**
* Manually review and correct predictions
* Iteratively retrain the model (active learning loop)

### 3. External Data Sources

* Leverage existing labeled datasets (e.g., satellite imagery datasets)
* Align and adapt them to your domain

### 4. Weak Supervision

* Use heuristic rules or metadata (e.g., map overlays, GIS data)
* Combine multiple weak signals to approximate labels

---

## Key Takeaways

* Without ground truth, evaluation relies heavily on **qualitative analysis**, which can reveal important failure modes but lacks precision.
* Performance gaps across classes are often driven by **data imbalance and feature representation issues**.
* Improving results typically requires a **data-centric approach**, complemented by targeted model and training adjustments.
* Building or approximating ground truth is essential for **robust, scalable evaluation and continuous improvement**.


## Summary

1. **Choose a model** with `list_timm_models` or the timm docs.
2. **Prepare paths and integer labels** (here, EuroSAT via Hugging Face → temp folders).
3. **Wrap data** in `RemoteSensingDataset` with the correct channel count.
4. **Train** with `train_timm_classifier` (full fine-tune, frozen backbone, optional `class_weights`).
5. **Infer** with `TimmClassifier.load_from_checkpoint` + `predict_with_timm`.

